# IT25101144 - K-Nearest Neighbors (KNN)

**AIML Group Assignment** | **Dataset:** UCI Default of Credit Card Clients

## Setup — shared train/test split

This notebook is a standalone copy of this member's section from `group_pipeline.ipynb`. The model code below is unchanged.

This cell loads the same 80/20 split that `group_pipeline.ipynb` saves to `results/outputs/train.csv` and `results/outputs/test.csv`, and defines the same `record_model()` function, so the scores match the group notebook. Run `group_pipeline.ipynb` first if these files are missing.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

TARGET = 'default payment next month'

# Same rows as the shared split in group_pipeline.ipynb.
# float_precision='round_trip' reads every value back exactly as it was saved.
train_df = pd.read_csv('../results/outputs/train.csv', float_precision='round_trip')
test_df = pd.read_csv('../results/outputs/test.csv', float_precision='round_trip')
X_train, y_train = train_df.drop(columns=TARGET), train_df[TARGET]
X_test, y_test = test_df.drop(columns=TARGET), test_df[TARGET]
print(f'Train: {len(train_df):,} rows | Test: {len(test_df):,} rows')

model_results = {}

def record_model(student_id, model_name, fitted_model):
    """Evaluate a fitted binary classifier on the shared test set."""
    predictions = fitted_model.predict(X_test)
    result = {
        'IT Number': student_id,
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, predictions),
        'Precision': precision_score(y_test, predictions, zero_division=0),
        'Recall': recall_score(y_test, predictions, zero_division=0),
        'F1': f1_score(y_test, predictions, zero_division=0),
        'ROC-AUC': 'N/A',
    }
    classes = list(getattr(fitted_model, 'classes_', []))
    if 1 in classes and hasattr(fitted_model, 'predict_proba'):
        scores = fitted_model.predict_proba(X_test)[:, classes.index(1)]
        result['ROC-AUC'] = roc_auc_score(y_test, scores)
    elif 1 in classes and hasattr(fitted_model, 'decision_function'):
        scores = fitted_model.decision_function(X_test)
        if len(classes) == 2 and classes.index(1) == 0:
            scores = -scores
        result['ROC-AUC'] = roc_auc_score(y_test, scores)
    model_results[student_id] = result
    return pd.Series(result)


### IT25101144 - K-Nearest Neighbors (KNN)

**Theoretical Justification & Model Suitability:**
- **Suitability:** Non-parametric instance-based classifier. Classifies defaulting clients based on geometric proximity in standardized financial behavioral space (borrowers with identical delinquency trajectories and credit usage cluster together).
- **Tuning Strategy:** Loops over neighborhood sizes ($K \in [1, 3, 5, 7, 9, 11, 15, 21]$), weight schemes (`uniform` vs `distance`), and distance metrics (`euclidean` vs `manhattan`) using 3-fold cross-validation.
- **Distance Metric & Scaling:** Demonstrates why Stage 5 (StandardScaler) is indispensable (unscaled balances would completely dominate Euclidean distances) and analyzes the trade-off of distance-weighted voting.


In [ ]:
# ==============================================================================
# IT25101144 — K-Nearest Neighbors (Neighborhood Evaluation)
# ==============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             ConfusionMatrixDisplay, RocCurveDisplay)

# ------------------------------------------------------------------------------
# 0. Sanity check: KNN requires standardized features (Stage 5 provides this)
# ------------------------------------------------------------------------------
print('Sanity check — mean ≈ 0, std ≈ 1 for numeric features?')
print(X_train.iloc[:, :5].agg(['mean', 'std']).round(3))

# ------------------------------------------------------------------------------
# 1. Expanded GridSearchCV: K + weights + distance metric
# ------------------------------------------------------------------------------
knn_params = {
    'n_neighbors': [1, 3, 5, 7, 9, 11, 15, 21],
    'weights':     ['uniform', 'distance'],
    'metric':      ['euclidean', 'manhattan']
}
knn_search = GridSearchCV(
    KNeighborsClassifier(n_jobs=-1),
    param_grid=knn_params, scoring='f1', cv=3, n_jobs=-1
)
knn_search.fit(X_train, y_train)
best_knn = knn_search.best_estimator_

print('\n=== IT25101144: KNN Optimization ===')
print(f'Optimal Parameters : {knn_search.best_params_}')
print(f'Training CV F1     : {knn_search.best_score_:.4f}')

# ------------------------------------------------------------------------------
# 2. FIX LEAKAGE — K-sweep using CROSS-VALIDATION on TRAINING data only
# ------------------------------------------------------------------------------
print('\n=== K-Sensitivity via 3-fold Cross-Validation (no test leakage) ===')
k_range = [1, 3, 5, 7, 9, 11, 15, 21]
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_means, cv_stds = [], []
for k in k_range:
    scores = cross_val_score(
        KNeighborsClassifier(n_neighbors=k, weights='distance',
                             metric='euclidean', n_jobs=-1),
        X_train, y_train, cv=cv, scoring='f1', n_jobs=-1
    )
    cv_means.append(scores.mean()); cv_stds.append(scores.std())
    print(f"  K={k:>2}  CV F1 = {scores.mean():.4f} ± {scores.std():.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(k_range, cv_means, yerr=cv_stds, fmt='o-', capsize=4,
            color='#e67e22', linewidth=2, label='CV F1 (mean ± std)')
ax.axvline(best_knn.n_neighbors, color='#27ae60', linestyle=':',
           label=f'Chosen K = {best_knn.n_neighbors}')
ax.set_xlabel('Number of Neighbors (K)'); ax.set_ylabel('CV F1 Score')
ax.set_title('IT25101144 KNN: Cross-Validated F1 vs K (Bias–Variance Tradeoff)',
             fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# ------------------------------------------------------------------------------
# 3. Inference-time cost (KNN's real computational cost is prediction, not fit)
# ------------------------------------------------------------------------------
t0 = time.time(); best_knn.fit(X_train, y_train); fit_time = time.time() - t0
t0 = time.time(); _ = best_knn.predict(X_test);    pred_time = time.time() - t0
print(f"\nFit time       : {fit_time:.3f} s (KNN stores training data)")
print(f"Predict time   : {pred_time:.3f} s for {len(X_test):,} test rows")
print(f"Per-query time : {pred_time/len(X_test)*1e6:.1f} µs "
      f"(O(n_train × d) per query)")

# ------------------------------------------------------------------------------
# 4. Inline metrics
# ------------------------------------------------------------------------------
y_pred_knn = best_knn.predict(X_test)
y_prob_knn = best_knn.predict_proba(X_test)[:, 1]

print('\n=== Best KNN: Test-Set Metrics ===')
print(f"Accuracy : {accuracy_score(y_test, y_pred_knn):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_knn):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_knn):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_knn):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_knn):.4f}")
print('\n' + classification_report(y_test, y_pred_knn,
                                    target_names=['No Default', 'Default']))

# ------------------------------------------------------------------------------
# 5. Confusion Matrix + ROC Curve
# ------------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_estimator(
    best_knn, X_test, y_test, cmap='Oranges',
    display_labels=['No Default', 'Default'], ax=ax)
ax.set_title('IT25101144 KNN: Test Confusion Matrix', fontweight='bold')
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(5, 4))
RocCurveDisplay.from_estimator(best_knn, X_test, y_test, ax=ax,
                               name=f'KNN (K={best_knn.n_neighbors})')
ax.plot([0, 1], [0, 1], '--', color='gray', label='Chance')
ax.legend(); ax.set_title('IT25101144 KNN: ROC Curve', fontweight='bold')
plt.tight_layout(); plt.show()

# ------------------------------------------------------------------------------
# 6. Register into shared scoreboard
# ------------------------------------------------------------------------------
record_model('IT25101144', 'K-Nearest Neighbors (KNN)', best_knn)
